In [1]:
import torch
from torch import nn

In [ ]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv_block = nn.Sequential(nn.Conv2d(in_channels, out_channels, bias = False, **kwargs),
                                        nn.BatchNorm2d(out_channels, eps = 0.001),
                                        nn.ReLU())

    def forward(self, x):
        x = self.conv_block(x)
        return x

class InceptionA(nn.Module): # Figure 5
    def __init__(self, in_channels, pool_features):
        super().__init__()

        self.branch1 = nn.Sequential(BasicConv2d(in_channels, 64, kernel_size = 1),
                                     BasicConv2d(64, 96, kernel_size = 3, padding = 1),
                                     BasicConv2d(96, 96, kernel_size = 3, padding = 1))

        self.branch2 = nn.Sequential(BasicConv2d(in_channels, 48, kernel_size = 1),
                                     BasicConv2d(48, 64, kernel_size = 3, padding = 1))

        self.branch3 = nn.Sequential(nn.AvgPool2d(kernel_size = 3, stride = 1, padding = 1),
                                    BasicConv2d(in_channels, pool_features, kernel_size = 1))

        self.branch4 = BasicConv2d(in_channels, 64, kernel_size = 1)

    def forward(self, x):
        x = [self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)]
        return torch.cat(x, 1)

class ReductionA(nn.Module): # Bottleneck 피하면서 grid size 줄이기
    def __init__(self, in_channels):
        super().__init__()

        self.branch1 = nn.Sequential(BasicConv2d(in_channels, 64, kernel_size = 1),
                                     BasicConv2d(64, 96, kernel_size = 3, padding = 1),
                                     BasicConv2d(96, 96, kernel_size = 3, stride = 2))
        
        self.branch2 = nn.Sequential(BasicConv2d(in_channels, 64, kernel_size = 1),
                                     BasicConv2d(64, 384, kernel_size = 3, stride = 2))

        self.branch3 = nn.